# Data Visualization: Seeing Your Data

## Why Visualize?

When you receive a new dataset, your first instinct might be to run `df.describe()` and look at summary statistics. That's useful, but it misses a lot. Consider Anscombe's Quartet - four datasets with nearly identical means, standard deviations, and correlations, but completely different shapes when plotted.

In quantitative finance, looking at your data visually helps you:
- **Spot anomalies** - a price spike of 1000% is obviously wrong, but `mean()` won't tell you that
- **Understand distributions** - are returns normally distributed? Fat-tailed?
- **See relationships** - do two assets move together? Is there a trend?

By the end of this lesson, you'll be able to:
- Create quick visualizations to explore new datasets
- Use different chart types for different questions
- Build multi-panel figures for reports and presentations
- Understand matplotlib's architecture so you can customize as needed

Let's get started.

**Reference / extension.** The live route and timing are in [the lesson guide](../readme.md).
 These expanded examples preserve the original material; they are not an additional required lecture.
 Run from a fresh kernel. The setup below locates the lesson folder from the repository root or either notebook folder.

In [ ]:
from pathlib import Path
import os
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
lesson_dir = course_root / 'Lectures' / 'Lecture 2'
os.chdir(lesson_dir)
(lesson_dir / 'outputs').mkdir(exist_ok=True)

## Setup

We'll use matplotlib (the foundational plotting library in Python) and pandas (which wraps matplotlib for convenience).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# This "magic command" makes plots appear inline in the notebook
%matplotlib inline

# Set a nice default style
plt.style.use('seaborn-v0_8-whitegrid')

# Make plots a bit larger by default
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Ready to visualize!")

## Our Dataset: Simulated Stock Data

Rather than relying on an API (which requires keys and can be rate-limited), we'll generate synthetic stock data. This gives us full control and makes the notebook reproducible.

We'll use **Geometric Brownian Motion (GBM)**, a standard model for simulating stock prices. While real stock prices don't follow GBM perfectly (returns have fatter tails and volatility clusters), it's a reasonable approximation for generating practice data.

In [ ]:
def generate_stock_data(ticker, start_price, volatility, drift, days=500, seed=None):
    """
    Generate realistic stock price data using geometric Brownian motion.
    
    Parameters:
    - ticker: Stock symbol
    - start_price: Initial stock price
    - volatility: Annual volatility (e.g., 0.25 for 25%)
    - drift: Annual expected return (e.g., 0.10 for 10%)
    - days: Number of trading days to generate
    - seed: Random seed for reproducibility
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Daily parameters (252 trading days per year)
    daily_vol = volatility / np.sqrt(252)
    daily_drift = drift / 252
    
    # Generate returns using GBM
    daily_returns = np.random.normal(daily_drift, daily_vol, days)
    
    # Convert to prices
    price_series = start_price * np.exp(np.cumsum(daily_returns))
    
    # Create realistic OHLCV data
    dates = pd.date_range(end='2024-12-06', periods=days, freq='B')  # Business days
    
    # Generate intraday variation for OHLC
    intraday_vol = daily_vol * 0.3
    
    df = pd.DataFrame({
        'open': price_series * (1 + np.random.normal(0, intraday_vol, days)),
        'high': price_series * (1 + np.abs(np.random.normal(0, intraday_vol * 2, days))),
        'low': price_series * (1 - np.abs(np.random.normal(0, intraday_vol * 2, days))),
        'close': price_series,
        'volume': np.random.randint(1_000_000, 50_000_000, days)
    }, index=dates)
    
    # Ensure high >= max(open, close) and low <= min(open, close)
    df['high'] = df[['open', 'high', 'close']].max(axis=1)
    df['low'] = df[['open', 'low', 'close']].min(axis=1)
    
    df['ticker'] = ticker
    return df

# Generate data for several stocks with different characteristics
stocks = {
    'TECH': generate_stock_data('TECH', 150, 0.35, 0.15, seed=42),    # High growth tech
    'BANK': generate_stock_data('BANK', 45, 0.20, 0.08, seed=43),     # Stable financial
    'ENERGY': generate_stock_data('ENERGY', 80, 0.40, 0.05, seed=44), # Volatile energy
    'RETAIL': generate_stock_data('RETAIL', 120, 0.25, 0.10, seed=45), # Mid-volatility retail
    'PHARMA': generate_stock_data('PHARMA', 200, 0.30, 0.12, seed=46)  # Healthcare/pharma
}

# Combine into one DataFrame for some analyses
df_all = pd.concat(stocks.values()).reset_index().rename(columns={'index': 'date'})

print(f"Generated {len(stocks)} stocks with {len(stocks['TECH'])} days of data each")
print(f"Date range: {df_all['date'].min().date()} to {df_all['date'].max().date()}")

In [ ]:
# Let's look at our TECH stock
stocks['TECH'].head(10)

---

# Part 1: Quick Wins with Pandas Plotting

Pandas has plotting built right into DataFrames and Series. This is your **first tool** when exploring new data - quick, easy, and surprisingly powerful.

## Your First Plot: Just Call `.plot()`

The simplest way to visualize a time series is to just... plot it.

In [ ]:
# Plot closing prices for TECH
stocks['TECH']['close'].plot()
plt.title('TECH Stock Price')
plt.ylabel('Price ($)')

**What just happened?**
- Pandas automatically used the DataFrame index (dates) for the x-axis
- It created a line chart by default (sensible for time series)
- We added a title and y-label with matplotlib commands

Let's try multiple columns:

In [ ]:
# Plot open, high, low, close together
stocks['TECH'][ ['open', 'high', 'low', 'close'] ].plot()
plt.title('TECH OHLC Prices')
plt.ylabel('Price ($)')

**Observation**: High and low create an "envelope" around open and close. This is expected - the high/low represent the trading range each day.

But this chart is messy. What if we just want to see the daily range?

In [ ]:
# Calculate and plot daily trading range as a percentage
tech = stocks['TECH'].copy()
tech['daily_range_pct'] = (tech['high'] - tech['low']) / tech['close'] * 100

tech['daily_range_pct'].plot()
plt.title('TECH Daily Trading Range')
plt.ylabel('Range (% of close)')
plt.axhline(y=tech['daily_range_pct'].mean(), color='r', linestyle='--', label='LT Avg')
plt.legend()

**Key Insight**: We can immediately see that:
1. Most days have a range of 1-3%
2. There are occasional spikes (high volatility days)
3. The red dashed line shows the average - useful for spotting unusual activity

This is **exploratory data analysis (EDA)** in action!

## Chart Types: Choosing the Right Visualization

Different questions require different charts. Here's a quick reference:

| Question | Chart Type | `kind=` parameter |
|----------|------------|-------------------|
| How does X change over time? | Line | `'line'` (default) |
| What's the distribution of X? | Histogram | `'hist'` |
| How do X and Y relate? | Scatter | `'scatter'` |
| Compare categories | Bar | `'bar'` or `'barh'` |
| Show proportions | Pie | `'pie'` |
| Distribution shape | KDE/Density | `'kde'` or `'density'` |
| Spread and outliers | Box | `'box'` |

### Distribution Analysis: Histograms and KDE

**The Question**: "What do daily returns look like for this stock?"

In [ ]:
# Calculate daily returns
tech['returns'] = tech['close'].pct_change(fill_method=None) * 100  # In percentage

# Histogram
tech['returns'].plot(kind='hist', bins=50, edgecolor='black', alpha=0.7)
plt.title('Distribution of TECH Daily Returns')
plt.xlabel('Daily Return (%)')
plt.ylabel('Frequency')

# Add a vertical line at zero
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)

**Key Observations**:
1. Returns are roughly centered around zero (slightly positive due to drift)
2. The distribution looks roughly normal (bell-shaped)
3. There might be some "fat tails" - extreme returns that occur more often than a normal distribution would predict

Let's overlay a density estimate to see the shape more clearly:

In [ ]:
# KDE (Kernel Density Estimate) - smooth version of histogram
fig, ax = plt.subplots()

# Histogram with density=True so y-axis matches KDE
tech['returns'].dropna().plot(kind='hist', bins=50, density=True, 
                               alpha=0.5, edgecolor='black', ax=ax)

# Overlay KDE
tech['returns'].dropna().plot(kind='kde', ax=ax, linewidth=2, color='red')

plt.title('TECH Returns: Histogram vs KDE')
plt.xlabel('Daily Return (%)')
plt.legend(['KDE', 'Histogram'])

### Relationship Analysis: Scatter Plots

**The Question**: "Are returns for TECH and BANK correlated?"

In [ ]:
# Calculate returns for both stocks
returns_df = pd.DataFrame({
    'TECH': stocks['TECH']['close'].pct_change(fill_method=None) * 100,
    'BANK': stocks['BANK']['close'].pct_change(fill_method=None) * 100,
    'ENERGY': stocks['ENERGY']['close'].pct_change(fill_method=None) * 100
}).dropna()

returns_df.head()

In [ ]:
# Scatter plot: TECH vs BANK returns
returns_df.plot(kind='scatter', x='TECH', y='BANK', alpha=0.5)
plt.title('TECH vs BANK Daily Returns')
plt.xlabel('TECH Return (%)')
plt.ylabel('BANK Return (%)')

# Add reference lines
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

**Interpretation**:
- Points in top-right and bottom-left quadrants = positive correlation (both go up/down together)
- Points in top-left and bottom-right = negative correlation
- Circular cloud = low correlation
- Elongated diagonal = high correlation

Let's quantify this:

In [ ]:
# Correlation matrix
print("Correlation Matrix:")
print(returns_df.corr().round(3))

### Comparison Analysis: Bar Charts

**The Question**: "How do average returns compare across stocks?"

In [ ]:
# Calculate summary statistics for each stock
summary = pd.DataFrame({
    'Mean Return (%)': [stocks[t]['close'].pct_change(fill_method=None).mean() * 100 for t in stocks],
    'Volatility (%)': [stocks[t]['close'].pct_change(fill_method=None).std() * 100 for t in stocks],
    'Current Price': [stocks[t]['close'].iloc[-1] for t in stocks]
}, index=stocks.keys())

summary

In [ ]:
# Bar chart of mean returns
summary['Mean Return (%)'].plot(kind='bar', color=['#2ecc71', '#3498db', '#e74c3c', '#9b59b6', '#f39c12'])
plt.title('Average Daily Returns by Stock')
plt.ylabel('Mean Daily Return (%)')
plt.xticks(rotation=45)
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

In [ ]:
# Side-by-side comparison: Return vs Risk
summary[ ['Mean Return (%)', 'Volatility (%)'] ].plot(kind='bar')
plt.title('Risk-Return Profile by Stock')
plt.ylabel('Percentage')
plt.xticks(rotation=45)
plt.legend(loc='upper right')

### Spread Analysis: Box Plots

Box plots show the distribution's quartiles and outliers - perfect for comparing volatility profiles.

In [ ]:
# Prepare returns for all stocks
all_returns = pd.DataFrame({
    ticker: stocks[ticker]['close'].pct_change(fill_method=None) * 100 
    for ticker in stocks
}).dropna()

# Box plot
all_returns.plot(kind='box')
plt.title('Return Distributions by Stock')
plt.ylabel('Daily Return (%)')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

**Reading a Box Plot**:
- **Box**: Middle 50% of data (Q1 to Q3)
- **Line in box**: Median
- **Whiskers**: Extend to 1.5x the interquartile range
- **Dots**: Outliers beyond the whiskers

**Insight**: ENERGY has the widest box and most outliers = highest volatility. BANK is the tightest = lowest volatility.

---

# Part 2: Understanding Matplotlib Architecture

Pandas plotting is great for quick exploration, but for publication-quality charts or complex layouts, you need to understand matplotlib directly.

**Two Ways to Use Matplotlib**:
1. **Procedural (pyplot)**: Quick and dirty, good for exploration
2. **Object-Oriented**: More control, better for complex charts

Let's learn both!

## The Procedural Way: `plt.plot()`

This is what we've been doing implicitly. It's simple but has limitations.

In [ ]:
# Procedural approach
plt.figure(figsize=(12, 5))

plt.plot(stocks['TECH'].index, stocks['TECH']['close'], label='TECH')
plt.plot(stocks['BANK'].index, stocks['BANK']['close'], label='BANK')

plt.title('Stock Prices Over Time')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)

### Subplots with Procedural Style

You can create multiple plots, but it gets awkward:

In [ ]:
plt.figure(figsize=(14, 5))

# First subplot
plt.subplot(1, 2, 1)  # 1 row, 2 columns, first plot
plt.plot(stocks['TECH']['close'])
plt.title('TECH')
plt.ylabel('Price ($)')

# Second subplot
plt.subplot(1, 2, 2)  # 1 row, 2 columns, second plot
plt.plot(stocks['BANK']['close'])
plt.title('BANK')

plt.tight_layout()  # Prevent overlap


**The Problem**: If you want to go back and modify the first plot after creating the second, you have to `plt.subplot(1, 2, 1)` again. This gets messy.

**Also**: Once the cell renders, the plot is "flushed" - you can't modify it in the next cell.

## The Object-Oriented Way: `fig, ax = plt.subplots()`

This approach gives you more control. You create figure and axes objects explicitly, then work with them directly. This is generally preferred for anything beyond quick one-off plots.

In [ ]:
# Create figure and axes objects
fig, ax = plt.subplots(figsize=(12, 5))

# Plot on the axes
ax.plot(stocks['TECH'].index, stocks['TECH']['close'], label='TECH', linewidth=2)
ax.plot(stocks['BANK'].index, stocks['BANK']['close'], label='BANK', linewidth=2)

# Customize using axes methods
ax.set_title('Stock Prices Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# The figure is returned - we can pass it around or save it
fig

**Why is this better?**

1. We have explicit references to `fig` and `ax`
2. We can modify them in subsequent cells
3. We can pass them to functions
4. We can save the figure programmatically

Let's see how this helps with subplots:

In [ ]:
# Create a 2x2 grid of subplots
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))

# axes is now a 2D array
print(f"axes shape: {axes.shape}")
print(f"axes type: {type(axes)}")

In [ ]:
# Fill in each subplot
# Top-left: TECH price
axes[0, 0].plot(stocks['TECH']['close'], color='#2ecc71')
axes[0, 0].set_title('TECH Stock Price')
axes[0, 0].set_ylabel('Price ($)')

# Top-right: TECH volume
axes[0, 1].bar(stocks['TECH'].index, stocks['TECH']['volume'], color='#3498db', alpha=0.7, width=1)
axes[0, 1].set_title('TECH Trading Volume')
axes[0, 1].set_ylabel('Volume')

# Bottom-left: TECH returns histogram
tech_returns = stocks['TECH']['close'].pct_change(fill_method=None).dropna() * 100
axes[1, 0].hist(tech_returns, bins=50, color='#e74c3c', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('TECH Return Distribution')
axes[1, 0].set_xlabel('Daily Return (%)')
axes[1, 0].axvline(x=0, color='black', linestyle='--')

# Bottom-right: TECH rolling volatility
rolling_vol = tech_returns.rolling(20).std()
axes[1, 1].plot(rolling_vol, color='#9b59b6')
axes[1, 1].set_title('TECH 20-Day Rolling Volatility')
axes[1, 1].set_ylabel('Volatility (%)')

# Add overall title
fig.suptitle('TECH Stock Analysis Dashboard', fontsize=16, fontweight='bold', y=1.02)

# Adjust spacing
plt.tight_layout()
fig

**The magic**: Because we kept references to `fig` and `axes`, we can still modify them:

In [ ]:
# Add an annotation to the price chart (top-left)
max_price = stocks['TECH']['close'].max()
max_date = stocks['TECH']['close'].idxmax()

axes[0, 0].annotate(f'Peak: ${max_price:.2f}', 
                    xy=(max_date, max_price),
                    xytext=(max_date - timedelta(days=50), max_price * 0.9),
                    arrowprops=dict(arrowstyle='->', color='red'),
                    fontsize=10, color='red')

fig

## Combining Pandas Plotting with OOP Matplotlib

The best of both worlds: use pandas' convenient `.plot()` but direct it to specific axes!

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Use pandas .plot() but specify ax=
stocks['TECH']['close'].plot(ax=axes[0], title='TECH', color='green')
stocks['BANK']['close'].plot(ax=axes[1], title='BANK', color='blue')
stocks['ENERGY']['close'].plot(ax=axes[2], title='ENERGY', color='red')

# Set common ylabel
for ax in axes:
    ax.set_ylabel('Price ($)')

plt.tight_layout()
fig

---

# Part 3: Advanced Visualization Techniques

Now that we understand the architecture, let's explore some powerful techniques for quantitative analysis.

## Scatter Matrix: See All Pairwise Relationships

When you have multiple variables, a scatter matrix shows every pairwise relationship at once. This is incredibly useful for spotting correlations.

In [ ]:
from pandas.plotting import scatter_matrix

# Prepare returns for all stocks
all_returns = pd.DataFrame({
    ticker: stocks[ticker]['close'].pct_change(fill_method=None) * 100 
    for ticker in ['TECH', 'BANK', 'ENERGY', 'RETAIL']
}).dropna()

# Create scatter matrix
fig = scatter_matrix(all_returns, figsize=(12, 12), diagonal='kde', alpha=0.5)
plt.suptitle('Pairwise Return Relationships', fontsize=14, y=1.02)

**Reading the Scatter Matrix**:
- **Diagonal**: Distribution of each variable (KDE in this case)
- **Off-diagonal**: Scatter plots of each pair
- Look for elongated patterns (correlation) vs circular clouds (no correlation)

This is a useful first step when exploring relationships in a new dataset with multiple variables.

## Time Series Analysis: Lag Plots and Autocorrelation

For time series data, we often want to know: **Is today's value related to yesterday's?**

In [ ]:
from pandas.plotting import lag_plot, autocorrelation_plot

# Lag plot: Today's return vs Yesterday's return
tech_returns = stocks['TECH']['close'].pct_change(fill_method=None).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lag plot
lag_plot(tech_returns, ax=axes[0])
axes[0].set_title('Lag Plot: Return(t) vs Return(t-1)')
axes[0].set_xlabel('Return(t-1)')
axes[0].set_ylabel('Return(t)')

# Autocorrelation plot
autocorrelation_plot(tech_returns, ax=axes[1])
axes[1].set_title('Autocorrelation Function')
axes[1].set_xlim(0, 50)  # Zoom into first 50 lags

plt.tight_layout()

**Interpretation**:
- **Lag plot (left)**: A circular cloud means low autocorrelation (today's return doesn't predict tomorrow's)
- **Autocorrelation plot (right)**: Shows correlation at each lag. Values within the shaded band are not statistically significant.

**Interpretation limit**: Sample autocorrelation does not establish a profitable strategy; estimation error, costs, and out-of-sample behavior still matter.

## Rolling Statistics: Trends Over Time

Rolling windows let us see how statistics evolve. This is essential for detecting regime changes.

In [ ]:
tech = stocks['TECH'].copy()
tech['returns'] = tech['close'].pct_change(fill_method=None)

# Calculate rolling statistics
tech['rolling_mean_20'] = tech['close'].rolling(20).mean()
tech['rolling_mean_50'] = tech['close'].rolling(50).mean()
tech['rolling_vol_20'] = tech['returns'].rolling(20).std() * np.sqrt(252) * 100  # Annualized

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Price with moving averages
axes[0].plot(tech.index, tech['close'], label='Price', alpha=0.7)
axes[0].plot(tech.index, tech['rolling_mean_20'], label='20-day MA', linewidth=2)
axes[0].plot(tech.index, tech['rolling_mean_50'], label='50-day MA', linewidth=2)
axes[0].set_title('TECH Price with Moving Averages')
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Rolling volatility
axes[1].fill_between(tech.index, 0, tech['rolling_vol_20'], alpha=0.5, color='red')
axes[1].plot(tech.index, tech['rolling_vol_20'], color='darkred', linewidth=1)
axes[1].set_title('TECH 20-Day Rolling Volatility (Annualized)')
axes[1].set_ylabel('Volatility (%)')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

## Saving Your Work

Professional deliverable: save high-quality figures for reports.

In [ ]:
# Create a publication-quality figure
fig, ax = plt.subplots(figsize=(10, 6))

for ticker, color in [('TECH', '#2ecc71'), ('BANK', '#3498db'), ('ENERGY', '#e74c3c')]:
    # Normalize prices to start at 100 for comparison
    normalized = stocks[ticker]['close'] / stocks[ticker]['close'].iloc[0] * 100
    ax.plot(normalized, label=ticker, linewidth=2, color=color)

ax.set_title('Normalized Stock Performance (Base = 100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Indexed Price', fontsize=12)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5)

# Save in multiple formats
fig.savefig('outputs/stock_comparison.png', dpi=300, bbox_inches='tight')
fig.savefig('outputs/stock_comparison.pdf', bbox_inches='tight')  # Vector format for publications

print("Saved as stock_comparison.png and stock_comparison.pdf")

---

# Summary: Key Takeaways

## 1. Why Visualize?
- **Exploration**: Understand your data before modeling
- **Quality Control**: Spot anomalies and errors
- **Communication**: Tell stories with data

## 2. Pandas Plotting Cheat Sheet
```python
df['col'].plot()                    # Line chart (default)
df['col'].plot(kind='hist')         # Histogram
df['col'].plot(kind='box')          # Box plot
df.plot(kind='scatter', x='A', y='B')  # Scatter
df.plot(kind='bar')                 # Bar chart
```

## 3. Matplotlib Architecture
```python
# OOP approach (recommended)
fig, ax = plt.subplots()            # Single plot
fig, axes = plt.subplots(2, 2)      # Grid of plots

ax.plot(x, y)                       # Plot on axes
ax.set_title('Title')               # Customize
fig.savefig('file.png')             # Save
```

## 4. Advanced Tools
```python
from pandas.plotting import scatter_matrix, lag_plot, autocorrelation_plot

scatter_matrix(df)                  # All pairwise relationships
lag_plot(series)                    # Autocorrelation visualization
autocorrelation_plot(series)        # ACF plot
```

## 5. Best Practices
- Always label axes and add titles
- Use consistent colors for the same variables
- Choose the right chart type for your question
- Save figures in high resolution for reports

---

# What's Next?

Now that you can visualize data, you're ready to **clean** it. In the next notebook, we'll tackle:

- Missing values: detection and handling strategies
- Outliers: when to remove, when to cap, when to investigate
- Duplicates: finding and removing them
- Text cleaning: fixing messy string data

These skills are essential because **garbage in = garbage out**. The best model in the world is useless if your data is dirty.

Continue with [cleaning fundamentals](5_data_cleaning_fundamentals.ipynb).